# Day 09 — LangGraph Fundamentals, State & Reducers (hands-on)

Companion notebook to [`../notes.md`](../notes.md). Builds a real, working LangGraph graph with two
nodes, runs it, and prints the resulting state to see both reducer types in action.

No API key needed — the "agent" and "tool" nodes here are deterministic Python functions standing in
for real LLM/tool calls, so we can focus entirely on the graph mechanics.

In [1]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

## 1. Define the state schema

Two fields, two different reducer behaviors:
- `messages` uses `add_messages` — a custom reducer that **appends** new messages to the list.
- `loop_count` has no custom reducer, so it uses the **default: replace**.

In [2]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # custom reducer: append
    loop_count: int                          # default reducer: replace

## 2. Define the nodes

Each node is just a function: read the state, do something, return updates. Here they're
deterministic stand-ins for "call the LLM" and "call a tool" — in a real graph these would be real
model/tool calls (Day 05).

In [3]:
def agent_node(state: AgentState) -> dict:
    count = state.get("loop_count", 0) + 1
    reply = AIMessage(content=f"(thinking step {count}) I looked at the question.")
    return {"messages": [reply], "loop_count": count}


def tool_node(state: AgentState) -> dict:
    result = AIMessage(content="(tool result) 3 flights found, cheapest is $420.")
    return {"messages": [result]}

## 3. Wire the graph together

`START -> agent -> tool -> END`. Day 10 adds branching and loops on top of this same idea.

In [4]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tool", tool_node)
graph_builder.add_edge(START, "agent")
graph_builder.add_edge("agent", "tool")
graph_builder.add_edge("tool", END)

graph = graph_builder.compile()

# LangGraph can render the graph structure itself -- handy for sanity-checking what you built.
print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | agent |    
  +-------+    
      *        
      *        
      *        
  +------+     
  | tool |     
  +------+     
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## 4. Run it and inspect the final state

In [5]:
initial_state = {
    "messages": [HumanMessage(content="Book me a flight to Delhi")],
    "loop_count": 0,
}

result = graph.invoke(initial_state)

print("Final loop_count:", result["loop_count"])
print("\nFull message history:")
for m in result["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

Final loop_count: 1

Full message history:
  [HumanMessage] Book me a flight to Delhi
  [AIMessage] (thinking step 1) I looked at the question.
  [AIMessage] (tool result) 3 flights found, cheapest is $420.


## 5. Seeing the two reducers side by side

- `messages` started with **1** message (the human's question). Two nodes each added one more. The
  final state has **3** messages — nothing was lost, because `add_messages` appends.
- `loop_count` — `agent_node` returned `loop_count=1`. The final state's `loop_count` is exactly `1`,
  not summed or accumulated across nodes — because the default reducer replaces.

In [6]:
print("messages: started with 1, ended with", len(result["messages"]), "-> append reducer")
print("loop_count: agent_node returned 1, final state has", result["loop_count"], "-> replace reducer")

messages: started with 1, ended with 3 -> append reducer
loop_count: agent_node returned 1, final state has 1 -> replace reducer


## Try it yourself

- Add a third node and edge, and watch `get_graph().draw_ascii()` change.
- Change `loop_count: int` to accumulate instead (e.g. give it a custom reducer using `operator.add`)
  and see how the final value changes with the exact same nodes.
- Have `agent_node` read `state["messages"]` and print how many messages it can already see — this is
  exactly how a real LLM node would use conversation history.